In [ ]:
import openai, json, os, requests

client = openai.OpenAI()

messages = []

In [ ]:
def get_popular_movies():
    link = os.getenv("MOVIE_API_URL") + "/movies"
    response = requests.get(link)

    return response.json()

def get_movie_details(id):
    link = f"{os.getenv('MOVIE_API_URL')}/movies/{id}/credits"
    response = requests.get(link)
    return response.json()

def get_similar_movies(id):
    link = f"{os.getenv('MOVIE_API_URL')}/movies/{id}/similar"
    response = requests.get(link)
    return response.json()

FUNCTION_MAP ={
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_similar_movies': get_similar_movies
}

In [ ]:
TOOLS = [
    {
        "type" : "function",
        "function" : {
            "name" : "get_popular_movies",
            "description" : "A function to get List popular movies",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type" : "function",
        "function" : {
            "name" : "get_movie_details",
            "description" : "Get movie details by movie id",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "id" : {
                        "type" : "integer",
                        "description" : "The ID of the movie to get details for."
                    }
                },
                "required" : ["id"]
            }   
        }
    },
    {
        "type" : "function",
        "function" : {
            "name" : "get_similar_movies",
            "description" : "Get similar movies for a movie by movie id",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "id" : {
                        "type" : "integer",
                        "description" : "The ID of the movie to get similar movies for."
                    }
                },
                "required" : ["id"]
            }   
        }
    }
]

In [ ]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message :ChatCompletionMessage):
    if message.tool_calls:
        messages.append({"role": "assistant", 
                         "content": message.content or "",
                         "tool_calls": [{
                             "id": tool_call.id,
                             "type" : "function",
                                "function" : {
                                    "name" : tool_call.function.name,
                                    "arguments" : tool_call.function.arguments
                                }
                         } for tool_call in message.tool_calls]})
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            
            print(f"Agent: [{function_name}({arguments.get('id', '')}) 호출]")

            messages.append({
                "role":"tool",
                "tool_call_id":tool_call.id,
                "name"  : function_name,
                "content": json.dumps(result)
            })
            
        call_ai()
    else :
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    ) 
    process_ai_response(response.choices[0].message)


In [ ]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()
    
    

In [ ]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()
    
    

In [ ]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()
    
    